# V2a-RSN 220127_F4_run2 — Attribution and Feature Sensitivity Notebook: switching-VAR

Prerequisites: identity-preprocessed raw windows and the dataset `shared/recording` checkpoint from preprocessing. This notebook fits a separate purged chronological outer-owned feature bundle.
This notebook reports predictor-feature sensitivity only; it does not identify a biological intervention.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused, failed, running, and missing checkpoints are reported explicitly.


In [ ]:
import sys
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]


In [ ]:
from omegaconf import OmegaConf
from effectome.connectivity import ConnectivityConfig

from notebooks._shared import (
    PROJECT_ROOT,
    make_run,
    print_notebook_header,
    print_stage_status,
    run_candidate_drivers,
    run_feature_sensitivity,
    run_owned_feature_artifacts,
)

DATASET_ID = "220127_F4_run2"
DATASET_OUTPUT_ID = "v2a-rsns"
RECORDING_ID = "220127_F4_run2"
RUN_ID = "reference"
METHOD = "switching_var"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
RECORDING_PATH = OUTPUT_ROOT / RUN_ID / "shared" / "stages" / "recording" / "artifact.pkl"
WINDOWS_PATH = OUTPUT_ROOT / RUN_ID / "shared" / "stages" / "windows" / "artifact.pkl"
BEHAVIOR_KEY = "continuous"
FIT_SCOPE_ID = f"{RUN_ID}/{METHOD}/sensitivity-outer-0"
MULTIPLICITY_FAMILY = (f"{BEHAVIOR_KEY}:feature-edit",)
connectivity_data = OmegaConf.to_container(
    OmegaConf.load(PROJECT_ROOT / "conf" / "connectivity" / f"{METHOD}.yaml"), resolve=True
)
CONNECTIVITY_CFG = ConnectivityConfig(**connectivity_data)
FORCE = False

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_notebook_header(
    run,
    title=f"{METHOD} attribution and feature sensitivity",
    expected_stages=[
        "recording_input",
        "outer_owned_features",
        "candidate_drivers",
        "surrogate_model",
        "feature_sensitivity",
    ],
    force=FORCE,
)


In [ ]:
owned = run_owned_feature_artifacts(
    run, windows_path=WINDOWS_PATH, connectivity_cfg=CONNECTIVITY_CFG,
    fit_scope_id=FIT_SCOPE_ID, evaluation_fraction=0.2, force=FORCE,
)
print(f"Training windows: {len(owned.fit_indices)}, held-out windows: {len(owned.evaluation_indices)}")

candidates = run_candidate_drivers(
    run,
    recording_path=RECORDING_PATH,
    behavior_key=BEHAVIOR_KEY,
    lag=1,
    n_folds=5,
    embargo=4,
    seed=42,
    matched_control_percentile=95.0,
    fit_indices=owned.fit_indices,
    evaluation_indices=owned.evaluation_indices,
    fit_scope_id=owned.fit_scope_id,
    force=FORCE,
)
top = candidates.scores[0]
print(top)
print_stage_status(run)


In [ ]:
sensitivity = run_feature_sensitivity(
    run,
    recording_path=RECORDING_PATH,
    behavior_key=BEHAVIOR_KEY,
    lag=1,
    ridge_alpha=1.0,
    n_folds=5,
    embargo=4,
    min_skill=0.0,
    scale=0.0,
    dose_scales=[1.0, 0.5, 0.0],
    top_k=1,
    endpoint="behavior",
    seed=42,
    fit_indices=owned.fit_indices,
    evaluation_indices=owned.evaluation_indices,
    fit_scope_id=owned.fit_scope_id,
    multiplicity_family=MULTIPLICITY_FAMILY,
    hypothesis_id=f"{BEHAVIOR_KEY}:feature-edit",
    min_absolute_skill=0.0,
    max_distribution_shift=5.0,
    n_controls=64,
    control_resolution_policy="unavailable",
    force=FORCE,
)
print(sensitivity)
print(sensitivity.analysis_kind)
print(sensitivity.biological_intervention_identified)
print_stage_status(run)
